# Stellar Classification: Stacked Ensembling & Threshold Tuning

This notebook executes prediction stacking using a Logistic Regression meta-learner and Nelder-Mead decision threshold optimization across LightGBM, XGBoost, and CatBoost predictions. This maximizes the cross-validation Balanced Accuracy score and generates the final predictions.

In [1]:
import numpy as np
import pandas as pd
from sklearn.metrics import balanced_accuracy_score
import os

# Target mapping variables
TARGET_MAP = {'GALAXY': 0, 'QSO': 1, 'STAR': 2}
INV_TARGET_MAP = {0: 'GALAXY', 1: 'QSO', 2: 'STAR'}

## 1. Directory Configurations & Dynamic Outputs

We configure dataset directories, prediction probability sources, and target output paths, ensuring portability across local and Kaggle environments.

In [2]:
# Dynamic path resolution for Local vs Kaggle environment
if os.path.exists('/kaggle/input/competitions/playground-series-s6e6'):
    DATA_DIR = '/kaggle/input/competitions/playground-series-s6e6'
    PRED_DIR = '/kaggle/input/notebooks/tuannm3812/stellar-classification-baseline-modeling/predictions'
    OUTPUT_DIR = '.'
else:
    DATA_DIR = '../data'
    PRED_DIR = '../predictions'
    OUTPUT_DIR = '..'

print(f"Data Directory: {DATA_DIR}")
print(f"Predictions Directory: {PRED_DIR}")
print(f"Output Directory: {OUTPUT_DIR}")

Data Directory: /kaggle/input/competitions/playground-series-s6e6
Predictions Directory: /kaggle/input/notebooks/tuannm3812/stellar-classification-baseline-modeling/predictions
Output Directory: .


## 2. Loading Probability Predictions & Target Labels

We load the ground-truth training classes alongside the saved out-of-fold validation and test prediction probability matrices. Stacking prediction probabilities (confidence scores) rather than discrete labels maintains calibration and improves class boundary separation.

In [3]:
# Load ground truth class labels
train_path = os.path.join(DATA_DIR, 'train.csv')
train = pd.read_csv(train_path)
y = train['class'].map(TARGET_MAP).values

# Load OOF prediction probabilities
lgb_oof = np.load(os.path.join(PRED_DIR, 'lgb_oof.npy'))
xgb_oof = np.load(os.path.join(PRED_DIR, 'xgb_oof.npy'))
cat_oof = np.load(os.path.join(PRED_DIR, 'cat_oof.npy'))

# Load test prediction probabilities
lgb_test = np.load(os.path.join(PRED_DIR, 'lgb_test.npy'))
xgb_test = np.load(os.path.join(PRED_DIR, 'xgb_test.npy'))
cat_test = np.load(os.path.join(PRED_DIR, 'cat_test.npy'))

print(f"Loaded shapes - y: {y.shape}, LGB OOF: {lgb_oof.shape}, XGB OOF: {xgb_oof.shape}, Cat OOF: {cat_oof.shape}")

Loaded shapes - y: (577347,), LGB OOF: (577347, 3), XGB OOF: (577347, 3), Cat OOF: (577347, 3)


## 3. Evaluate Baseline Models

### 3.1. Individual Model Scores
We verify the baseline Balanced Accuracy scores of the individual models before ensembling. This establishes the baseline performance threshold that our ensemble optimization must exceed.

In [4]:
lgb_score = balanced_accuracy_score(y, np.argmax(lgb_oof, axis=1))
xgb_score = balanced_accuracy_score(y, np.argmax(xgb_oof, axis=1))
cat_score = balanced_accuracy_score(y, np.argmax(cat_oof, axis=1))

print("--- Individual Model Scores (Balanced Accuracy) ---")
print(f"LightGBM: {lgb_score:.5f}")
print(f"XGBoost:  {xgb_score:.5f}")
print(f"CatBoost: {cat_score:.5f}")

--- Individual Model Scores (Balanced Accuracy) ---
LightGBM: 0.96512
XGBoost:  0.96536
CatBoost: 0.96211


### 3.2. Model Disagreement Analysis
Ensembling is only effective if the component models make *different* errors. We isolate the disagreement region—rows where our models do not predict the same class—and print the pairwise agreement percentages.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Convert probabilities to hard class labels
lgb_preds = np.argmax(lgb_oof, axis=1)
xgb_preds = np.argmax(xgb_oof, axis=1)
cat_preds = np.argmax(cat_oof, axis=1)

preds_matrix = np.column_stack([lgb_preds, xgb_preds, cat_preds])
model_names = ['LightGBM', 'XGBoost', 'CatBoost']

# Calculate disagreement rows
unanimous = np.all(preds_matrix == preds_matrix[:, [0]], axis=1)
disagree = ~unanimous
print(f"Total training rows:  {len(y)}")
print(f"Unanimous agreement:  {unanimous.sum()} ({unanimous.mean()*100:.2f}%)")
print(f"Disagreement region:  {disagree.sum()} ({disagree.mean()*100:.2f}%)\n")

# Pairwise agreement matrix
agree = np.zeros((3, 3))
for a in range(3):
    for b in range(3):
        agree[a, b] = (preds_matrix[:, a] == preds_matrix[:, b]).mean() * 100

plt.figure(figsize=(6, 5))
sns.heatmap(pd.DataFrame(agree, index=model_names, columns=model_names), 
            annot=True, fmt='.2f', cmap='viridis', square=True, cbar_kws={'label': 'Agreement %'})
plt.title('Pairwise Model Agreement Heatmap')
plt.tight_layout()
plt.show()

## 4. Stacking & Threshold Tuning

### 4.1. Logistic Regression Stacking
Instead of manual grid-search probability blending, we implement a **Logistic Regression meta-learner** to stack model probabilities. We train the meta-model on OOF prediction vectors using 5-fold Stratified CV to predict target classes, then fit it on the full dataset to generate final test probabilities.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_predict, StratifiedKFold

# Stack predictions: shape (N, 3 * 3 = 9)
Xoof = np.hstack([lgb_oof, xgb_oof, cat_oof])
Xtest = np.hstack([lgb_test, xgb_test, cat_test])

# Initialize Logistic Regression Meta-Learner
meta_model = LogisticRegression(max_iter=2000, C=1.0, multi_class='multinomial', class_weight='balanced', random_state=42)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Generate leakage-free OOF stacked predictions
print("--- Training Stacking Meta-Learner ---")
meta_oof = cross_val_predict(meta_model, Xoof, y, cv=skf, method='predict_proba')

# Evaluate Meta-Learner OOF Balanced Accuracy
stacked_score = balanced_accuracy_score(y, np.argmax(meta_oof, axis=1))
print(f"Stacked Meta-Learner OOF Balanced Accuracy: {stacked_score:.5f}")

# Fit on full OOF predictions to generate test probabilities
meta_model.fit(Xoof, y)
stacked_test_prob = meta_model.predict_proba(Xtest)

### 4.2. Nelder-Mead Threshold Optimization
Standard inference assigns classes using $\arg\max(P_c)$. However, when optimizing for Balanced Accuracy (unweighted average recall) under severe class skew, the standard probability boundaries are mathematically sub-optimal. We search for class-specific multipliers using the **Nelder-Mead simplex algorithm** (via `scipy.optimize.minimize`) to directly maximize Balanced Accuracy on our Out-of-Fold (OOF) predictions.

In [ ]:
from scipy.optimize import minimize

def optimize_thresholds(oof_probs, y_true):
    def loss_func(weights):
        if np.any(weights <= 0.01):
            return 999.0 + np.sum(np.abs(weights))
        scaled_probs = oof_probs * weights
        preds = np.argmax(scaled_probs, axis=1)
        return -balanced_accuracy_score(y_true, preds)
    
    init_weights = [1.0, 1.0, 1.0]
    res = minimize(loss_func, init_weights, method='Nelder-Mead', options={'maxiter': 500})
    best_weights = res.x / np.sum(res.x) * 3.0
    best_score = -res.fun
    return best_weights, best_score, res.success, res.message

print("--- Optimizing Decision Thresholds over Stacked Predictions ---")
best_thresholds, optimized_score, opt_success, opt_msg = optimize_thresholds(meta_oof, y)
if not opt_success:
    print(f"Warning: Nelder-Mead threshold optimization did not converge: {opt_msg}")

t_gal, t_qso, t_star = best_thresholds

print(f"Original Stacking Score: {stacked_score:.5f}")
print(f"Optimized Stacking Score: {optimized_score:.5f}")
print(f"Optimal Multipliers -> GALAXY: {t_gal:.4f}, QSO: {t_qso:.4f}, STAR: {t_star:.4f}")

## 5. Final Inference & Submission

### 5.1. Test Class Scaling
We apply the optimized threshold multipliers to the stacked test probabilities from the Logistic Regression meta-learner prior to argmax classification.

### 5.2. Generate Submission File
We save the class predictions to `submission.csv` using the inverse target mapping, formatting the output for final leaderboard submission.

In [ ]:
# Applying stacked test predictions from Logistic Regression meta-learner
final_test_prob = stacked_test_prob

# Applying optimized threshold multipliers before taking argmax
scaled_test_prob = final_test_prob * best_thresholds
final_preds = np.argmax(scaled_test_prob, axis=1)

# Create submission DataFrame
sub_sample_path = os.path.join(DATA_DIR, 'sample_submission.csv')
submission = pd.read_csv(sub_sample_path)
submission['class'] = pd.Series(final_preds).map(INV_TARGET_MAP)

# Save to CSV
sub_out_path = os.path.join(OUTPUT_DIR, 'submission.csv')
submission.to_csv(sub_out_path, index=False)
print(f"Submission file created successfully at {sub_out_path}!")
print(submission['class'].value_counts())